## Acknowledgements

To start, I’d like to thank **Tong Hui Kang** and **konbu17**.

The CoT data and part of the hyperparameter settings used in this notebook are adapted from Tong Hui Kang’s open-source GitHub repository, while the training code is modified based on konbu17’s public notebook.  
- [Tong Hui Kang’s open-source GitHub repository](https://github.com/tonghuikang/nemotron)
- [konbu17’s public notebook](https://www.kaggle.com/code/konbu17/nemotron-sft-lora-with-cot)

## Overview

In this notebook, I will try to reproduce the method released by Tong Hui Kang and train a model that can reach around **0.85** on the public leaderboard.

In this version, training on **"lm_head"** in target_modules has been **removed**. At the same time, the microbatch size has been reset to **1**. In my experiments, increasing the microbatch size to 2 reduced training time by about **3 hours**, but it also caused the LB score to drop by around **0.1**, and I have not yet been able to address this issue through hyperparameter tuning.

## Notes on Training

Setting per_device_train_batch_size to 2 and gradient_accumulation_steps to 16 can reduce the training time to around **4 hours** while maintaining similar performance **(0.84–0.85)**. However, given the nature of the competition, a score drop of about 0.1 is still difficult to accept.

It may be possible to restore training performance comparable to that of a microbatch size of 1 through **further hyperparameter tuning**.

I hope this notebook can also help others participate more effectively in this competition.

## Mode Selection

In [ ]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================

import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/dgxchen/trained-adapter"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# ============================================================
# V39 — V34 (LB 0.85) + bit in-format verify/self-correct augmentation
# ============================================================
# V34 = Muon + dgxchen verbatim bit + km cryptarithm swap -> LB 0.85 (val_950 89.5%, bit 93.5%)
# V39 = V34 with bit CoT replaced by the SAME THK CoT + appended self-verification
#       (and ~30% self-correction trajectories). Format = THK original (avoids v4 0.76 crash).
#       augmented_bit.csv: 1754 rows (1660 augmented / 94 kept verbatim), 99.9% boxed==gold.
#
# SINGLE VARIABLE vs V34: only the bit_manipulation CoT content changes.
#   V34 bit = dgxchen baseline 1754 rows (THK format, 93.5% val_950)
#   V39 bit = same rows + "Verify ... reproduces every example" section
#             + ~30% "Double-check bit j ... MISMATCH ... re-derive" self-correction
#   Everything else identical: Muon @ 2e-3, standard CE, km cryptarithm swap, 1 epoch.
#
# Data composition (= V34's 8189 rows exactly):
#   DROP cryptarithm_{deduce,guess}: dgxchen 100% default-to-concat WRONG (V34 dropped these)
#   DROP bit_manipulation: baseline dgxchen bit (replaced by augmented version)
#   ADD kimberleyduran_cryptarithm (1140 rows, V34 used)
#   ADD augmented_bit (1754 rows, THK + verify/self-correct)
#   KEEP dgxchen equation/cipher/gravity/numeral/unit
# Effective: 7830 - 781 - 1754 + 1140 + 1754 = 8189 training rows
#
# GATE: bit val_950 must beat 93.5% AND other 8 classes must not regress -> only then submit LB.
#       Retreat = V34 = 0.85 (#53080019).

SYNTH_DROP_TYPES = ['cryptarithm_deduce', 'cryptarithm_guess', 'bit_manipulation']
SYNTH_ADD_GLOBS = [
    'kimberleyduran_cryptarithm.csv',
    'augmented_bit.csv',
]

# Training layer (cell 11) - same as V34 verified (LB 0.85):
#   1. Muon optimizer @ LR 2e-3 (pkuszboi +0.01 LB)
#   2. Standard CE loss
#   3. warmup_ratio=0.0
#   4. lm_head auto-injected (Unsloth MoE handles)

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})


## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec：", importlib.util.find_spec("triton"))


In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


In [ ]:
# trl installation is handled by the Unsloth offline setup cell below.
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install/import here; the Unsloth setup cell will install compatible packages.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    _unsloth_wheels = recursive_wheels("unsloth-*.whl")
    if not _unsloth_wheels:
        raise FileNotFoundError("unsloth wheel not found under /kaggle/input")
    packages_dir = os.path.dirname(_unsloth_wheels[0])
    print("Resolved packages_dir:", packages_dir)
    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime because Nemotron depends on CUDA wheels.")

    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels):
        return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel = pick_last(all_mamba)
    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("Could not find a compatible mamba_ssm wheel under /kaggle/input.")

    print("Offline package installation finished. Restart the kernel if Kaggle keeps stale imports from earlier runs.")
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm / unsloth installation.")


In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear

    LORA_RANK = 32
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.0
    # v30: "lm_head" added so saved adapter_config.json reflects it (vLLM
    # cross-checks at inference). Manual injection below is the actual workaround
    # because Unsloth silently drops lm_head from MoE LoRA wrapping.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj", "up_proj", "down_proj",
        "lm_head",  # v30 single-variable delta
    ]

    print("Creating trainable LoRA wrapper via FastLanguageModel.get_peft_model ...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    model.print_trainable_parameters()

    # ── v30 single-variable change: manually inject lm_head LoRA ──
    # Unsloth silently drops lm_head LoRA on MoE models (huikang docs this in
    # end-to-end-finetuning-for-lb-0-85 notebook). Even listing "lm_head" in
    # target_modules above is ignored by FastLanguageModel.get_peft_model.
    # We add it back manually here — this is the ONLY delta from v23 baseline.
    _causal_lm = model
    while hasattr(_causal_lm, "model"):
        _causal_lm = _causal_lm.model
    _lm_head = _causal_lm.lm_head
    if not isinstance(_lm_head, LoraLinear):
        _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            _cfg,
            "default",
            target=_lm_head,
            target_name="lm_head",
            parent=_causal_lm,
        )
        print("Manually added LoRA to lm_head (v30 single-variable delta)")
    else:
        print("lm_head already has LoRA (skipped manual injection)")

    print("Trainable params AFTER lm_head injection:")
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping trainable LoRA construction.")


## Mode A: Train on Kaggle

In [ ]:
if TRAIN_ON_KAGGLE:
    import pandas as pd
    import random
    import gc, time
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    import torch
    import torch.nn as nn

    SEED = 42
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    import glob as _glob
    _cot_csvs = _glob.glob("/kaggle/input/**/problem_ids_matched.csv", recursive=True)
    if not _cot_csvs:
        raise FileNotFoundError("problem_ids_matched.csv not found under /kaggle/input")
    DATASET_PATH = _cot_csvs[0]
    print("Resolved DATASET_PATH:", DATASET_PATH)
    df = pd.read_csv(DATASET_PATH)
    print(f"Full dataset: {len(df)} rows")

    # --- synthetic data mixing (configured in the Mode Selection cell) ---
    REQUIRED_COLS = {"id", "prompt", "answer", "type", "generated_cot"}
    print("Baseline composition:", dict(sorted(df["type"].value_counts().items())))
    if SYNTH_DROP_TYPES:
        _before = len(df)
        df = df[~df["type"].isin(SYNTH_DROP_TYPES)].reset_index(drop=True)
        print(f"Dropped {_before - len(df)} baseline rows of types {SYNTH_DROP_TYPES}")
    for _pat in SYNTH_ADD_GLOBS:
        _hits = sorted(_glob.glob(f"/kaggle/input/**/{_pat}", recursive=True))
        if not _hits:
            raise FileNotFoundError(f"SYNTH_ADD_GLOBS pattern matched nothing: {_pat}")
        for _p in _hits:
            _extra = pd.read_csv(_p)
            _missing = REQUIRED_COLS - set(_extra.columns)
            if _missing:
                raise ValueError(f"{_p} missing columns: {_missing}")
            _extra = _extra[list(df.columns)]  # align column order
            df = pd.concat([df, _extra], ignore_index=True)
            print(f"Added {len(_extra)} synthetic rows from {_p}")
    print(f"Final dataset after mixing: {len(df)} rows")
    print("Final composition:", dict(sorted(df["type"].value_counts().items())))


    train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"Full dataset: {len(df)} rows")


    import re
    import math
    from collections import defaultdict
    from torch.utils.data import DataLoader, Sampler
    records = []
    record_types = []
    for _, row in train_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = str(row["generated_cot"])
        cat = str(row["type"])
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            continue
        user_content = prompt + PROMPT_SUFFIX
        # v34: all tasks get \boxed{answer} wrapping (no sub-skill exceptions)
        cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
        assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
        records.append({"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})
        record_types.append(cat)
    dataset = HFDataset.from_list(records)
    print(f"SFT records: {len(records)}")

    def formatting_prompts_func(example):
        messages = example["messages"]
        if messages and isinstance(messages[0], dict):
            conversations = [messages]
        else:
            conversations = messages

        texts = []
        for conversation in conversations:
            try:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                    enable_thinking=True,
                )
            except TypeError:
                text = tokenizer.apply_chat_template(
                    conversation,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            texts.append(text)
        return texts


    # --- mid-training checkpoint + progress printing ---
    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    os.makedirs(ADAPTER_DIR, exist_ok=True)
    SAVE_EVERY = 40  # optimizer steps

    class PeriodicAdapterSave(TrainerCallback):
        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step and state.global_step % SAVE_EVERY == 0:
                m = kwargs.get("model")
                if m is not None:
                    m.save_pretrained(ADAPTER_DIR)
                    print(f"[checkpoint] adapter saved at step {state.global_step}", flush=True)

    class PrintProgress(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and state.global_step and "loss" in logs:
                print(f"[step {state.global_step}/{state.max_steps}]  "
                      f"loss={logs['loss']:.4f}  lr={logs.get('learning_rate', 0):.2e}  "
                      f"epoch={state.epoch:.2f}", flush=True)

    # v34: Muon @ LR 2e-3 + standard CE + warmup_ratio=0.0 (pkuszboi config)
    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-3,                      # Muon scale (pkuszboi/Bobber verified)
        lr_scheduler_type="linear",
        warmup_ratio=0.0,                         # v33 FIX: override TRL default 0.1
        max_length=8192,
        adam_beta1=0.9,                           # unused (Muon overrides optimizer)
        adam_beta2=0.95,
        adam_epsilon=1e-8,
        weight_decay=0.0,
        max_grad_norm=1e9,                        # no clip (huikang/pkuszboi)
        logging_steps=10,
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to="none",
        packing=False,
    )

    def build_stratified_index_order(labels, batch_size, seed):
        """Approximate nemotron-master's stratified batching over effective batches."""
        by_label = defaultdict(list)
        for idx, label in enumerate(labels):
            by_label[label].append(idx)

        rng = random.Random(seed)
        for idx_list in by_label.values():
            rng.shuffle(idx_list)

        n_batches = max(1, math.ceil(len(labels) / batch_size))
        batches = [[] for _ in range(n_batches)]
        batch_order = list(range(n_batches))
        rng.shuffle(batch_order)

        assigned = 0
        for label in sorted(by_label.keys()):
            for idx in by_label[label]:
                batches[batch_order[assigned % n_batches]].append(idx)
                assigned += 1

        order = [idx for batch in batches for idx in batch]
        if len(order) != len(labels):
            raise ValueError("Stratified order size mismatch")
        return order

    class PrecomputedOrderSampler(Sampler):
        def __init__(self, order):
            self.order = list(order)

        def __iter__(self):
            return iter(self.order)

        def __len__(self):
            return len(self.order)

    # ──────────────────────────────────────────────────────────
    # v34: Muon optimizer (pkuszboi/Bobber verified: +0.01 LB)
    # ──────────────────────────────────────────────────────────
    @torch.compile()
    def zeropower_via_newtonschulz5(G, steps: int = 5, eps: float = 1e-7):
        a, b, c = (3.4445, -4.7750, 2.0315)
        X = G
        X /= X.norm() + eps
        transposed = G.size(0) > G.size(1)
        if transposed:
            X = X.T
        for _ in range(steps):
            A = X @ X.T
            B = b * A + c * A @ A
            X = a * X + B @ X
        return X.T if transposed else X

    class Muon(torch.optim.Optimizer):
        def __init__(self, params, lr: float, momentum: float, backend_steps: int, nesterov: bool = True):
            super().__init__(
                params,
                dict(lr=lr, momentum=momentum, backend_steps=backend_steps, nesterov=nesterov),
            )

        @torch.no_grad()
        def step(self, closure=None):
            loss = None
            if closure is not None:
                with torch.enable_grad():
                    loss = closure()

            for group in self.param_groups:
                params = group["params"]
                if not params:
                    continue
                lr = group["lr"]
                momentum = group["momentum"]
                backend_steps = group["backend_steps"]
                nesterov = group["nesterov"]

                for p in params:
                    if p.grad is None:
                        continue
                    if p.ndim < 2:
                        continue
                    g = p.grad
                    state = self.state[p]
                    if "momentum_buffer" not in state:
                        state["momentum_buffer"] = torch.zeros_like(g)
                    buf = state["momentum_buffer"]
                    buf.mul_(momentum).add_(g)
                    if nesterov:
                        g = g.add(buf, alpha=momentum)
                    g = zeropower_via_newtonschulz5(g, steps=backend_steps)
                    g *= max(1, g.size(0) / g.size(1)) ** 0.5
                    p.add_(g, alpha=-lr)

            return loss

    class StratifiedSFTTrainer(SFTTrainer):
        def __init__(self, *args, stratified_order=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.stratified_order = stratified_order

        def get_train_dataloader(self):
            if self.train_dataset is None:
                raise ValueError("Trainer requires a train_dataset.")
            if self.stratified_order is None:
                return super().get_train_dataloader()
            if len(self.stratified_order) != len(self.train_dataset):
                raise ValueError("Stratified order length does not match train dataset")

            dataloader_kwargs = {
                "batch_size": self.args.per_device_train_batch_size,
                "sampler": PrecomputedOrderSampler(self.stratified_order),
                "collate_fn": self.data_collator,
                "num_workers": self.args.dataloader_num_workers,
                "pin_memory": self.args.dataloader_pin_memory,
                "persistent_workers": self.args.dataloader_persistent_workers,
                "drop_last": self.args.dataloader_drop_last,
            }
            if self.args.dataloader_num_workers > 0:
                dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor

            return DataLoader(self.train_dataset, **dataloader_kwargs)

    class MuonStratifiedTrainer(StratifiedSFTTrainer):
        """v34 = Stratified batching + Muon optimizer + Standard CE loss.

        pkuszboi 0.85 verified path: Muon + standard CE + dgxchen base.
        V34 single variable: swap cryptarithm dgxchen → kimberleyduran (Bobber 验证 +0.01 LB).
        """
        def __init__(self, *args, muon_lr=2e-3, **kwargs):
            super().__init__(*args, **kwargs)
            self.muon_lr = muon_lr
            print(f"[MuonStratifiedTrainer] muon_lr={self.muon_lr}")

        def create_optimizer(self):
            if self.optimizer is None:
                print("[MuonStratifiedTrainer] creating Muon optimizer for LoRA params")
                matrix_params = [
                    p for name, p in self.model.named_parameters()
                    if p.requires_grad and "lora" in name and p.ndim >= 2
                ]
                print(f"[MuonStratifiedTrainer] Muon param count: {len(matrix_params)}")
                self.optimizer = Muon(
                    matrix_params,
                    lr=self.muon_lr,
                    momentum=0.95,
                    backend_steps=5,
                )
            return self.optimizer

        # v34: standard CE loss — no branch weighting

    effective_batch_size = max(
        1,
        training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    )
    stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
    print(f"Approx stratified effective batch size: {effective_batch_size}")
    print("Stratified batching by type:", dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

    trainer = MuonStratifiedTrainer(
        muon_lr=2e-3,
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
        formatting_func=formatting_prompts_func,
        stratified_order=stratified_order,
        callbacks=[PrintProgress(), PeriodicAdapterSave()],
    )

    total_steps = math.ceil(len(dataset) / effective_batch_size)
    print(f"Expected optimizer steps for 1 epoch: ~{total_steps}")
    print("Starting V34 SFT training (Muon + standard CE + kimberleyduran cryptarithm swap)...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training done in {elapsed/60:.1f} min")


    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")

## Mode B: Load Pre-trained LoRA

In [ ]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")
